In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import math
from dataclasses import dataclass

# ── RMSNorm ─────────────────────────────────────
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        x_f = x.float()
        norm = x_f * torch.rsqrt(x_f.pow(2).mean(-1, keepdim=True) + self.eps)
        return (norm * self.scale).to(x.dtype)

# ── Attention ───────────────────────────────────
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0

        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=False)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=False)

        self.n_head = config.n_head
        self.n_embd = config.n_embd

    def forward(self, x):
        B, T, C = x.size()

        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)

        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

# ── MLP ─────────────────────────────────────────
class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=False)
        self.proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=False)

    def forward(self, x):
        return self.proj(F.gelu(self.fc(x)))

# ── Block ───────────────────────────────────────
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = RMSNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln2 = RMSNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

# ── Config ──────────────────────────────────────
@dataclass
class VidhiConfig:
    block_size: int = 512
    vocab_size: int = 50304
    n_layer: int = 12
    n_head: int = 8
    n_embd: int = 512

# ── Model ───────────────────────────────────────
class Vidhi(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        self.wpe = nn.Embedding(config.block_size, config.n_embd)

        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layer)])
        self.ln_f = RMSNorm(config.n_embd)

        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.wte.weight = self.lm_head.weight  # weight tying

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()

        pos = torch.arange(0, T, device=idx.device)
        x = self.wte(idx) + self.wpe(pos)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            return logits, None

        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            targets.view(-1)
        )

        return logits, loss

In [3]:
import torch

# ─────────────────────────────────────────────
# 1. Device
# ─────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"

# ─────────────────────────────────────────────
# 2. Create Model
# ─────────────────────────────────────────────
config = VidhiConfig()

model = Vidhi(config)


# ─────────────────────────────────────────────
# 3. Path from Kaggle Input
# ─────────────────────────────────────────────
ckpt_path = "/content/vidhi.pt"


# ─────────────────────────────────────────────
# 4. Load Weights
# ─────────────────────────────────────────────
state_dict = torch.load(
    ckpt_path,
    map_location=device
)

model.load_state_dict(state_dict)

model.to(device)
model.eval()

print("✅ Vidhi model loaded successfully")

✅ Vidhi model loaded successfully


In [4]:
from datasets import load_dataset
import torch
import torch.nn.functional as F
import tiktoken
import numpy as np
import os

# ─────────────────────────────────────────────
# 1. Setup
# ─────────────────────────────────────────────
device = "cuda"

block_size = 512
batch_size = 4
grad_accum = 8
max_iters = 3000

lr = 2e-5
min_lr = 1e-6
warmup = 100

enc = tiktoken.get_encoding("gpt2")

def encode(text):
    return enc.encode_ordinary(text)

# ─────────────────────────────────────────────
# 2. Load PubMedQA
# ─────────────────────────────────────────────
ds = load_dataset("pubmed_qa", "pqa_labeled", split="train")

# ─────────────────────────────────────────────
# 3. Format examples
# ─────────────────────────────────────────────
def format_example(ex):

    question = ex["question"]

    context = " ".join(ex["context"]["contexts"])

    long_answer = ex["long_answer"]

    final_decision = ex["final_decision"]

    text = f"""Question:
{question}

Context:
{context}

Explanation:
{long_answer}

Answer:
{final_decision}
"""

    return {"text": text}

ds = ds.map(format_example)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

pqa_labeled/train-00000-of-00001.parquet:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [5]:
# ─────────────────────────────────────────────
# 4. Tokenize
# ─────────────────────────────────────────────
def tokenize(ex):

    ids = encode(ex["text"])

    ids = ids[:block_size]

    return {"ids": ids}

ds = ds.map(tokenize)

ds = ds.filter(lambda x: len(x["ids"]) > 50)

# ─────────────────────────────────────────────
# 6. Optimizer
# ─────────────────────────────────────────────
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=lr,
    weight_decay=0.01
)

# ─────────────────────────────────────────────
# 7. Batch function
# ─────────────────────────────────────────────
def get_batch():

    batch = ds.shuffle().select(range(batch_size))

    x_list = []
    y_list = []

    for sample in batch:

        ids = sample["ids"]

        x = ids[:-1]
        y = ids[1:]

        pad_len = block_size - len(x)

        x += [0] * pad_len
        y += [-1] * pad_len

        x_list.append(torch.tensor(x))
        y_list.append(torch.tensor(y))

    return (
        torch.stack(x_list).to(device),
        torch.stack(y_list).to(device)
    )

# ─────────────────────────────────────────────
# 8. LR schedule
# ─────────────────────────────────────────────
def get_lr(it):

    if it < warmup:
        return lr * it / warmup

    decay = (it - warmup) / (max_iters - warmup)

    coeff = 0.5 * (1 + np.cos(np.pi * decay))

    return min_lr + coeff * (lr - min_lr)

# ─────────────────────────────────────────────
# 9. Training loop
# ─────────────────────────────────────────────
best_loss = float("inf")

os.makedirs("reasoning_ckpt", exist_ok=True)

for it in range(max_iters):

    lr_now = get_lr(it)

    for g in optimizer.param_groups:
        g["lr"] = lr_now

    x, y = get_batch()

    logits, _ = model(x)

    loss = F.cross_entropy(
        logits.view(-1, logits.size(-1)),
        y.view(-1),
        ignore_index=-1
    )

    loss = loss / grad_accum

    loss.backward()

    if (it + 1) % grad_accum == 0:

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()
        optimizer.zero_grad()

    if it % 100 == 0:

        print(
            f"iter {it} | "
            f"loss {loss.item()*grad_accum:.4f}"
        )

        if loss.item() < best_loss:

            best_loss = loss.item()

            torch.save(
                model.state_dict(),
                "reasoning_ckpt/vidhi_reasoning.pt"
            )

            print("💾 Saved best reasoning model")

print("✅ Reasoning fine-tuning complete")

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

iter 0 | loss 4.1194
💾 Saved best reasoning model
iter 100 | loss 3.5895
💾 Saved best reasoning model
iter 200 | loss 3.0125
💾 Saved best reasoning model
iter 300 | loss 3.3544
iter 400 | loss 2.9541
💾 Saved best reasoning model
iter 500 | loss 3.1778
iter 600 | loss 2.9746
iter 700 | loss 2.9123
💾 Saved best reasoning model
iter 800 | loss 2.9641
iter 900 | loss 2.6600
💾 Saved best reasoning model
iter 1000 | loss 3.2522
iter 1100 | loss 2.4931
💾 Saved best reasoning model
iter 1200 | loss 2.9303
iter 1300 | loss 2.3454
💾 Saved best reasoning model
iter 1400 | loss 2.6772
iter 1500 | loss 2.4937
iter 1600 | loss 2.7320
iter 1700 | loss 2.4012
iter 1800 | loss 2.6732
iter 1900 | loss 2.5753
iter 2000 | loss 2.3867
iter 2100 | loss 2.4895
iter 2200 | loss 2.5662
iter 2300 | loss 2.7819
iter 2400 | loss 2.7047
iter 2500 | loss 2.6627
iter 2600 | loss 2.6940
iter 2700 | loss 2.3999
iter 2800 | loss 2.2872
💾 Saved best reasoning model
iter 2900 | loss 2.2030
💾 Saved best reasoning model
✅ 

In [6]:
import torch
import tiktoken

# ─────────────────────────────────────────────
# 1. Setup
# ─────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"

enc = tiktoken.get_encoding("gpt2")

def encode(text):
    return enc.encode_ordinary(text)

def decode(tokens):
    return enc.decode(tokens)

# ─────────────────────────────────────────────
# 2. Load Vidhi
# ─────────────────────────────────────────────
config = VidhiConfig()

model = Vidhi(config)

ckpt_path = "reasoning_ckpt/vidhi_reasoning.pt"

state_dict = torch.load(
    ckpt_path,
    map_location=device
)

model.load_state_dict(state_dict)

model.to(device)
model.eval()

print("✅ Reasoning-tuned Vidhi loaded")

# ─────────────────────────────────────────────
# 3. Generation function
# ─────────────────────────────────────────────
@torch.no_grad()
def generate(
    prompt,
    max_new_tokens=180,
    temperature=0.7,
    top_k=40,
    top_p=0.9,
    repetition_penalty=1.1
):

    input_ids = torch.tensor(
        [encode(prompt)],
        dtype=torch.long
    ).to(device)

    for _ in range(max_new_tokens):

        idx_cond = input_ids[:, -512:]

        logits, _ = model(idx_cond)

        logits = logits[:, -1, :]

        # repetition penalty
        for token in set(input_ids[0].tolist()):
            logits[:, token] /= repetition_penalty

        # temperature
        logits = logits / temperature

        # top-k
        if top_k is not None:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = -float("inf")

        # top-p
        sorted_logits, sorted_indices = torch.sort(
            logits,
            descending=True
        )

        cumulative_probs = torch.cumsum(
            torch.softmax(sorted_logits, dim=-1),
            dim=-1
        )

        sorted_indices_to_remove = cumulative_probs > top_p
        sorted_indices_to_remove[:, 1:] = \
            sorted_indices_to_remove[:, :-1].clone()

        sorted_indices_to_remove[:, 0] = False

        for i in range(sorted_indices.size(1)):
            if sorted_indices_to_remove[0, i]:
                logits[0, sorted_indices[0, i]] = -float("inf")

        probs = torch.softmax(logits, dim=-1)

        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        input_ids = torch.cat(
            [input_ids, next_token],
            dim=1
        )

    return decode(input_ids[0].tolist())

# ─────────────────────────────────────────────
# 4. Helper for pretty testing
# ─────────────────────────────────────────────
def ask(question):

    prompt = f"""Question:
{question}

Explanation:
"""

    print("\n🩺 Question:")
    print(question)

    print("\n📘 Vidhi:")
    print("─" * 60)

    output = generate(prompt)

    print(output)

    print("─" * 60)

# ─────────────────────────────────────────────
# 5. Test Questions
# ─────────────────────────────────────────────

ask("Why is Metformin considered first-line therapy for Type 2 Diabetes?")

ask("Explain the pathophysiology of dengue shock syndrome.")

ask("Why are ACE inhibitors used in patients with hypertension and diabetes?")


✅ Reasoning-tuned Vidhi loaded

🩺 Question:
Why is Metformin considered first-line therapy for Type 2 Diabetes?

📘 Vidhi:
────────────────────────────────────────────────────────────
Question:
Why is Metformin considered first-line therapy for Type 2 Diabetes?

Explanation:
The aim of this study was to determine the use of Metformin, a new antiandrogen in diabetes mellitus. A prospective cross-sectional study was performed among 40 patients with Type 1 diabetes and 20 healthy subjects. Metformin was measured by measuring the total number of metformin sites, using an electronic microcomputer. There were no significant differences between diabetic and non-diabetic subjects. In non-diabetic subjects Metformin has been reported as an alternative to placebo or metformin in diabetic subjects. However, Metformin is not found to be associated with any significant difference in the results of treatment for Type 1 diabetes, but it is not clear whether Metformin could be used in combination with 

In [7]:
import torch
import torch.nn.functional as F
from datasets import load_dataset
import tiktoken
import numpy as np
import os

# ─────────────────────────────────────────────
# 1. Setup
# ─────────────────────────────────────────────
device = "cuda"

block_size = 512
batch_size = 4
grad_accum = 8
max_iters = 3000

lr = 2e-5
min_lr = 1e-6
warmup = 100

enc = tiktoken.get_encoding("gpt2")

def encode(text):
    return enc.encode_ordinary(text)

# ─────────────────────────────────────────────
# 2. Load MedInstruct
# ─────────────────────────────────────────────
ds = load_dataset(
    "medalpaca/medical_meadow_medical_flashcards",
    split="train"
)

# ─────────────────────────────────────────────
# 3. Format as CHAT
# ─────────────────────────────────────────────
def format_example(ex):

    instruction = ex["input"]
    response = ex["output"]

    text = f"""User:
{instruction}

Assistant:
{response}
"""

    return {"text": text}

ds = ds.map(format_example)

# ─────────────────────────────────────────────
# 4. Tokenization
# ─────────────────────────────────────────────
def tokenize(ex):

    ids = encode(ex["text"])

    ids = ids[:block_size]

    return {"ids": ids}

ds = ds.map(tokenize)

ds = ds.filter(lambda x: len(x["ids"]) > 30)

# ─────────────────────────────────────────────
# 5. Load Vidhi
# ─────────────────────────────────────────────
config = VidhiConfig()

model = Vidhi(config)

# load BEST pretrained model
ckpt = torch.load(
    "reasoning_ckpt/vidhi_reasoning.pt",
    map_location=device
)

model.load_state_dict(ckpt)

model.to(device)

print("✅ Loaded Vidhi")

# ─────────────────────────────────────────────
# 6. Optimizer
# ─────────────────────────────────────────────
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=lr,
    weight_decay=0.01
)

# ─────────────────────────────────────────────
# 7. Batch function
# ─────────────────────────────────────────────
def get_batch():

    batch = ds.shuffle().select(range(batch_size))

    x_list = []
    y_list = []

    for sample in batch:

        ids = sample["ids"]

        x = ids[:-1]
        y = ids[1:]

        pad_len = block_size - len(x)

        x += [0] * pad_len
        y += [-1] * pad_len

        x_list.append(torch.tensor(x))
        y_list.append(torch.tensor(y))

    return (
        torch.stack(x_list).to(device),
        torch.stack(y_list).to(device)
    )

# ─────────────────────────────────────────────
# 8. LR Schedule
# ─────────────────────────────────────────────
def get_lr(it):

    if it < warmup:
        return lr * it / warmup

    decay = (it - warmup) / (max_iters - warmup)

    coeff = 0.5 * (1 + np.cos(np.pi * decay))

    return min_lr + coeff * (lr - min_lr)

# ─────────────────────────────────────────────
# 9. Training Loop
# ─────────────────────────────────────────────
best_loss = float("inf")

os.makedirs("instruction_ckpt", exist_ok=True)

print("\n🚀 Starting instruction tuning...")

for it in range(max_iters):

    lr_now = get_lr(it)

    for g in optimizer.param_groups:
        g["lr"] = lr_now

    x, y = get_batch()

    logits, _ = model(x)

    loss = F.cross_entropy(
        logits.view(-1, logits.size(-1)),
        y.view(-1),
        ignore_index=-1
    )

    loss = loss / grad_accum

    loss.backward()

    if (it + 1) % grad_accum == 0:

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()
        optimizer.zero_grad()

    # ── Logging ────────────────────────────
    if it % 100 == 0:

        print(
            f"iter {it} | "
            f"loss {loss.item()*grad_accum:.4f}"
        )

        if loss.item() < best_loss:

            best_loss = loss.item()

            torch.save(
                model.state_dict(),
                "instruction_ckpt/vidhi_medassist.pt"
            )

            print("💾 Saved best instruction-tuned model")

print("\n✅ Instruction tuning complete!")

README.md: 0.00B [00:00, ?B/s]

medical_meadow_wikidoc_medical_flashcard(…):   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/33955 [00:00<?, ? examples/s]

Map:   0%|          | 0/33955 [00:00<?, ? examples/s]

Map:   0%|          | 0/33955 [00:00<?, ? examples/s]

Filter:   0%|          | 0/33955 [00:00<?, ? examples/s]

✅ Loaded Vidhi

🚀 Starting instruction tuning...
iter 0 | loss 2.8844
💾 Saved best instruction-tuned model
iter 100 | loss 2.8453
💾 Saved best instruction-tuned model
iter 200 | loss 2.7329
💾 Saved best instruction-tuned model
iter 300 | loss 2.3485
💾 Saved best instruction-tuned model
iter 400 | loss 2.0180
💾 Saved best instruction-tuned model
iter 500 | loss 2.3803
iter 600 | loss 2.0823
iter 700 | loss 1.9489
💾 Saved best instruction-tuned model
iter 800 | loss 2.3757
iter 900 | loss 2.0291
iter 1000 | loss 2.0534
iter 1100 | loss 2.1420
iter 1200 | loss 2.4207
iter 1300 | loss 2.4939
iter 1400 | loss 1.7772
💾 Saved best instruction-tuned model
iter 1500 | loss 2.2177
iter 1600 | loss 2.2188
iter 1700 | loss 1.9021
iter 1800 | loss 2.0437
iter 1900 | loss 2.7222
iter 2000 | loss 2.5010
iter 2100 | loss 2.2272
iter 2200 | loss 1.7406
💾 Saved best instruction-tuned model
iter 2300 | loss 2.1941
iter 2400 | loss 2.1090
iter 2500 | loss 2.0749
iter 2600 | loss 1.7963
iter 2700 | loss 2.

In [8]:
import torch
import tiktoken

# ─────────────────────────────────────────────
# 1. Setup
# ─────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"

enc = tiktoken.get_encoding("gpt2")

def encode(text):
    return enc.encode_ordinary(text)

def decode(tokens):
    return enc.decode(tokens)

# ─────────────────────────────────────────────
# 2. Load Instruction-Tuned Vidhi
# ─────────────────────────────────────────────
config = VidhiConfig()

model = Vidhi(config)

ckpt_path = "instruction_ckpt/vidhi_medassist.pt"

state_dict = torch.load(
    ckpt_path,
    map_location=device
)

model.load_state_dict(state_dict)

model.to(device)
model.eval()

print("✅ Instruction-tuned Vidhi loaded")

# ─────────────────────────────────────────────
# 3. Generation Function
# ─────────────────────────────────────────────
@torch.no_grad()
def generate(
    prompt,
    max_new_tokens=180,
    temperature=0.65,
    top_k=40,
    top_p=0.9,
    repetition_penalty=1.15
):

    input_ids = torch.tensor(
        [encode(prompt)],
        dtype=torch.long
    ).to(device)

    for _ in range(max_new_tokens):

        idx_cond = input_ids[:, -512:]

        logits, _ = model(idx_cond)

        logits = logits[:, -1, :]

        # repetition penalty
        for token in set(input_ids[0].tolist()):
            logits[:, token] /= repetition_penalty

        # temperature
        logits = logits / temperature

        # top-k
        if top_k is not None:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = -float("inf")

        # top-p
        sorted_logits, sorted_indices = torch.sort(
            logits,
            descending=True
        )

        cumulative_probs = torch.cumsum(
            torch.softmax(sorted_logits, dim=-1),
            dim=-1
        )

        sorted_indices_to_remove = cumulative_probs > top_p

        sorted_indices_to_remove[:, 1:] = \
            sorted_indices_to_remove[:, :-1].clone()

        sorted_indices_to_remove[:, 0] = False

        for i in range(sorted_indices.size(1)):

            if sorted_indices_to_remove[0, i]:
                logits[0, sorted_indices[0, i]] = -float("inf")

        probs = torch.softmax(logits, dim=-1)

        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        input_ids = torch.cat(
            [input_ids, next_token],
            dim=1
        )

    return decode(input_ids[0].tolist())

# ─────────────────────────────────────────────
# 4. Assistant Interface
# ─────────────────────────────────────────────
def ask(question):

    prompt = f"""User:
{question}

Assistant:
"""

    print("\n🩺 User:")
    print(question)

    print("\n🤖 Vidhi:")
    print("─" * 60)

    output = generate(prompt)

    # remove prompt from output
    response = output[len(prompt):]

    print(response.strip())

    print("─" * 60)

# ─────────────────────────────────────────────
# 5. Test Questions
# ─────────────────────────────────────────────

ask("Why is Metformin considered first-line therapy for Type 2 Diabetes?")

ask("Explain the pathophysiology of dengue shock syndrome.")

ask("Why are ACE inhibitors beneficial in diabetic patients with hypertension?")

ask("What are the common symptoms of iron deficiency anemia?")


✅ Instruction-tuned Vidhi loaded

🩺 User:
Why is Metformin considered first-line therapy for Type 2 Diabetes?

🤖 Vidhi:
────────────────────────────────────────────────────────────
Metformin is considered first-line treatment for Type 1 diabetes.

Somatostatin is a type of medication that is used to treat diabetes mellitus and other types of diabetes, including diabetic retinopathy, and the risk associated with this treatment can be reduced by reducing the risk of complications such as diabetes and cardiovascular disease. This treatment may also result in side effects or adverse effects on insulin levels, such as hypertension and heart failure. It is important for individuals who have not yet received metformin to receive it at high risk for developing these symptoms, such as those who are still receiving metformin and without medication. In addition to its potential benefits, metformin should include medications to increase blood glucose levels, and drugs to prevent complications such